# 03. Python Analysis — SaaS Customer Behavior

This notebook performs deeper behavioral analysis of the RavenStack SaaS dataset using Python and PySpark.

The SQL analysis established the main business KPIs, revenue trends, churn rates, and customer-level patterns.

Python analysis will focus on understanding **why customers may churn** by examining:

- Feature usage and customer engagement
- Support activity and satisfaction
- Customer behavior differences between churned and retained customers
- Customer segments and churn patterns
- Potential indicators of churn risk

The objective is to generate validated business insights that can later support the Power BI dashboard.

In [2]:
from pyspark.sql import functions as F

# Load source tables from the RavenStack Lakehouse

accounts = spark.table("accounts")
subscriptions = spark.table("subscriptions")
feature_usage = spark.table("feature_usage")
support_tickets = spark.table("support_tickets")
churn_events = spark.table("churn_events")

print("Tables loaded successfully")

print("Accounts:", accounts.count())
print("Subscriptions:", subscriptions.count())
print("Feature Usage:", feature_usage.count())
print("Support Tickets:", support_tickets.count())
print("Churn Events:", churn_events.count())

StatementMeta(, 76b53e30-46d3-4735-a804-8843b5835bc4, 4, Finished, Available, Finished, False)

Tables loaded successfully
Accounts: 500
Subscriptions: 5000
Feature Usage: 25000
Support Tickets: 2000
Churn Events: 600


### Result

All required RavenStack SaaS datasets were successfully loaded from the Fabric Lakehouse.

- **Accounts:** 500 records
- **Subscriptions:** 5,000 records
- **Feature Usage:** 25,000 records
- **Support Tickets:** 2,000 records
- **Churn Events:** 600 records

The datasets are now ready for behavioral and churn analysis using PySpark and Python.

## 1. Feature Usage vs Churn

This analysis examines whether product engagement differs between churned and retained customers.

We will compare:

- Average feature usage
- Total usage activity
- Number of active usage days

The objective is to identify whether lower product engagement is associated with customer churn.

In [3]:
# Aggregate feature usage at the subscription level

usage_summary = (
    feature_usage
    .groupBy("subscription_id")
    .agg(
        F.sum("usage_count").alias("total_usage_count"),
        F.count("*").alias("total_usage_events"),
        F.countDistinct("usage_date").alias("active_usage_days")
    )
)

# Connect subscriptions to accounts and churn status

subscription_churn = (
    subscriptions
    .select(
        "subscription_id",
        "account_id",
        "churn_flag"
    )
    .withColumn(
        "churned",
        F.when(F.col("churn_flag") == True, 1).otherwise(0)
    )
)

# Compare feature usage between churned and retained subscriptions

usage_churn_analysis = (
    usage_summary
    .join(
        subscription_churn,
        on="subscription_id",
        how="inner"
    )
    .groupBy("churned")
    .agg(
        F.countDistinct("account_id").alias("customers"),
        F.round(F.avg("total_usage_count"), 2).alias("avg_usage_count"),
        F.round(F.avg("total_usage_events"), 2).alias("avg_usage_events"),
        F.round(F.avg("active_usage_days"), 2).alias("avg_active_usage_days")
    )
    .orderBy("churned")
)

display(usage_churn_analysis)

StatementMeta(, 7daffc0c-6948-41c1-bf29-dcb14590e809, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3df58817-728c-462e-94be-872cd50b28e4)

### Result

- Retained customers showed slightly higher product engagement than churned customers.
- Average usage count was **50.56** for retained customers compared with **49.29** for churned customers.
- Average usage events were **5.05** for retained customers compared with **4.91** for churned customers.
- Average active usage days were **5.03** for retained customers compared with **4.90** for churned customers.
- The differences are relatively small, suggesting that overall feature usage alone may not be a strong indicator of churn.

Further analysis of usage patterns, support activity, and customer characteristics is required to identify stronger churn-risk signals.

## 2. Support Activity & Satisfaction vs Churn

This analysis examines whether customer support experience is associated with customer churn.

We will compare churned and retained customers based on:

- Average number of support tickets
- Average ticket resolution time
- Average first response time
- Average customer satisfaction score
- Average number of escalated tickets

The objective is to identify whether poor support experiences or higher support activity may be associated with customer churn.

In [4]:
# Aggregate support activity at the account level

support_summary = (
    support_tickets
    .groupBy("account_id")
    .agg(
        F.count("*").alias("total_tickets"),
        F.round(F.avg("resolution_time_hours"), 2).alias("avg_resolution_time"),
        F.round(F.avg("first_response_time_minutes"), 2).alias("avg_first_response_time"),
        F.round(F.avg("satisfaction_score"), 2).alias("avg_satisfaction_score"),
        F.sum(
            F.when(F.col("escalation_flag") == True, 1).otherwise(0)
        ).alias("escalated_tickets")
    )
)

# Get account-level churn status

account_churn = (
    accounts
    .select(
        "account_id",
        "churn_flag"
    )
    .withColumn(
        "churned",
        F.when(F.col("churn_flag") == True, 1).otherwise(0)
    )
)

# Compare support experience between retained and churned customers

support_churn_analysis = (
    support_summary
    .join(
        account_churn,
        on="account_id",
        how="inner"
    )
    .groupBy("churned")
    .agg(
        F.countDistinct("account_id").alias("customers"),
        F.round(F.avg("total_tickets"), 2).alias("avg_tickets"),
        F.round(F.avg("avg_resolution_time"), 2).alias("avg_resolution_time"),
        F.round(F.avg("avg_first_response_time"), 2).alias("avg_first_response_time"),
        F.round(F.avg("avg_satisfaction_score"), 2).alias("avg_satisfaction_score"),
        F.round(F.avg("escalated_tickets"), 2).alias("avg_escalated_tickets")
    )
    .orderBy("churned")
)

display(support_churn_analysis)

StatementMeta(, 7daffc0c-6948-41c1-bf29-dcb14590e809, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e41a1534-d0a5-4f05-bef8-d3c966eeb3eb)

### Result

- Retained customers averaged **4.08 support tickets**, while churned customers averaged **4.00 tickets**.
- Average resolution time was **36.45 hours** for retained customers compared with **35.49 hours** for churned customers.
- Average first response time was **89.58 minutes** for retained customers compared with **84.93 minutes** for churned customers.
- Average satisfaction score was **3.95** for retained customers compared with **4.00** for churned customers.
- Churned customers had a slightly higher average number of escalated tickets (**0.22**) compared with retained customers (**0.18**).
- Overall, support activity and satisfaction show **only small differences** between churned and retained customers.
- The slightly higher escalation rate among churned customers may indicate a potential relationship between difficult support interactions and churn, but further analysis is required to establish a stronger pattern.

### Business Insight

Support experience alone does not appear to be a strong churn indicator in this analysis. However, **ticket escalations may be worth investigating further** alongside product engagement, customer tenure, plan tier, and other customer characteristics.


## 3. Customer Engagement Patterns

This analysis examines customer engagement behavior using product usage data.

We will analyze:

- Most frequently used features
- Usage activity across features
- Average usage count by feature
- Beta feature adoption
- Usage patterns of different features

The objective is to understand which product features drive engagement and identify potential differences in feature adoption that may be relevant to customer retention.

In [5]:
# Analyze usage patterns by feature

feature_engagement = (
    feature_usage
    .groupBy("feature_name")
    .agg(
        F.count("*").alias("usage_events"),
        F.sum("usage_count").alias("total_usage_count"),
        F.round(F.avg("usage_count"), 2).alias("avg_usage_count"),
        F.countDistinct("subscription_id").alias("active_subscriptions")
    )
    .orderBy(F.desc("total_usage_count"))
)

display(feature_engagement)

StatementMeta(, 7daffc0c-6948-41c1-bf29-dcb14590e809, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, be8216cc-82eb-4f41-a75a-7491ed1c8566)

### Result

- The dataset contains **40 product features** with relatively consistent usage patterns.
- **feature_32** recorded the highest total usage count with **6,686** usage events/counts.
- **feature_15** followed with **6,621**, while **feature_6** recorded **6,546**.
- The average usage count per event remained relatively consistent across features, ranging from approximately **9.80 to 10.35**.
- **feature_12** had the highest number of active subscriptions at **624**, indicating broad adoption.
- **feature_23** had the lowest number of active subscriptions at **533** among the features shown in the analysis.
- Overall, feature adoption is fairly evenly distributed, with no single feature dominating usage by a very large margin.

### Business Insight

Customer engagement is spread across multiple product features rather than being concentrated around one feature. This suggests that churn-risk analysis should consider **overall engagement and feature adoption patterns** rather than relying on usage of a single feature.


## 4. Customer Segments & Churn Patterns

This analysis examines whether churn patterns differ across customer segments.

We will compare churn across:

- Industry
- Plan tier
- Number of seats
- Referral source

The objective is to identify customer segments with relatively higher churn and understand which customer characteristics may be associated with customer retention.

In [6]:
# Create account-level customer segments with churn status

customer_segments = (
    accounts
    .select(
        "account_id",
        "industry",
        "plan_tier",
        "seats",
        "referral_source",
        "churn_flag"
    )
    .withColumn(
        "churned",
        F.when(F.col("churn_flag") == True, 1).otherwise(0)
    )
)

# Churn by industry

churn_by_industry = (
    customer_segments
    .groupBy("industry")
    .agg(
        F.countDistinct("account_id").alias("customers"),
        F.sum("churned").alias("churned_customers"),
        F.round(F.avg("churned") * 100, 2).alias("churn_rate")
    )
    .orderBy(F.desc("churn_rate"))
)

display(churn_by_industry)

StatementMeta(, 7daffc0c-6948-41c1-bf29-dcb14590e809, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 26ee35d6-ad22-4ab4-a7d7-e840f1876b91)

### Result

- **DevTools** has the highest churn rate at **30.97%**, with 35 out of 113 customers churned.
- **FinTech** has a churn rate of **22.32%**, with 25 out of 112 customers churned.
- **HealthTech** has a churn rate of **21.88%**, with 21 out of 96 customers churned.
- **EdTech** has a churn rate of **16.46%**, with 13 out of 79 customers churned.
- **Cybersecurity** has the lowest churn rate at **16.00%**, with 16 out of 100 customers churned.

### Business Insight

Churn varies considerably across industries. **DevTools customers show the highest churn rate**, while **Cybersecurity customers show the lowest**. This suggests that industry segment may be an important factor to investigate when understanding customer retention.

However, industry alone does not establish the cause of churn. Further analysis of plan tier, seats, referral source, and customer engagement is required.

### 4.2 Churn by Plan Tier

This analysis examines whether customer churn differs across subscription plan tiers.

We will compare:

- Number of customers
- Number of churned customers
- Churn rate

The objective is to identify whether certain plan tiers have higher churn and may require closer investigation for customer retention.

In [7]:
# Churn by plan tier

churn_by_plan = (
    customer_segments
    .groupBy("plan_tier")
    .agg(
        F.countDistinct("account_id").alias("customers"),
        F.sum("churned").alias("churned_customers"),
        F.round(F.avg("churned") * 100, 2).alias("churn_rate")
    )
    .orderBy(F.desc("churn_rate"))
)

display(churn_by_plan)

StatementMeta(, 7daffc0c-6948-41c1-bf29-dcb14590e809, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fa5608e6-04ec-4f12-bbc5-50f8a132ab95)

### Result

- **Enterprise** customers have the highest churn rate at **22.08%**, with 34 out of 154 customers churned.
- **Basic** customers have a churn rate of **22.02%**, with 37 out of 168 customers churned.
- **Pro** customers have the lowest churn rate at **21.91%**, with 39 out of 178 customers churned.
- The churn rates across all three plan tiers are **very similar**, with less than a 0.2 percentage-point difference.

### Business Insight

Plan tier does not appear to be a strong differentiating factor for customer churn in this analysis. Since churn is almost evenly distributed across Enterprise, Basic, and Pro customers, other factors such as **industry, customer engagement, support experience, and acquisition source** should be investigated further.

### 4.3 Churn by Referral Source

This analysis examines whether customer churn differs based on how customers were acquired.

We will compare:

- Referral source
- Number of customers
- Number of churned customers
- Churn rate

The objective is to identify whether certain acquisition channels are associated with higher customer churn and may require further investigation.

In [8]:
# Churn by referral source

churn_by_referral = (
    customer_segments
    .groupBy("referral_source")
    .agg(
        F.countDistinct("account_id").alias("customers"),
        F.sum("churned").alias("churned_customers"),
        F.round(F.avg("churned") * 100, 2).alias("churn_rate")
    )
    .orderBy(F.desc("churn_rate"))
)

display(churn_by_referral)

StatementMeta(, 7daffc0c-6948-41c1-bf29-dcb14590e809, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9b98f739-f2ea-47f8-8383-cdbe15d56ad3)

### Result

- **Event** has the highest churn rate at **30.21%**, with 29 out of 96 customers churned.
- **Other** referral sources have a churn rate of **24.27%**, with 25 out of 103 customers churned.
- **Ads** have a churn rate of **23.47%**, with 23 out of 98 customers churned.
- **Organic** customers have a churn rate of **17.54%**, with 20 out of 114 customers churned.
- **Partner** referrals have the lowest churn rate at **14.61%**, with 13 out of 89 customers churned.

### Business Insight

Churn varies noticeably across acquisition channels. Customers acquired through **event-based sources show the highest churn**, while **partner referrals show the lowest churn**.

This suggests that acquisition source may be associated with customer retention. The business could investigate whether partner-referred customers have better onboarding, stronger customer fit, or different engagement patterns.

### 4.4 Churn by Number of Seats

This analysis examines whether customer size, represented by the number of seats, is associated with customer churn.

We will group customers into seat ranges and compare:

- Number of customers
- Number of churned customers
- Churn rate

The objective is to identify whether smaller or larger customer accounts show different churn patterns.

In [9]:
# Create customer size segments based on number of seats

customer_seat_segments = (
    customer_segments
    .withColumn(
        "seat_range",
        F.when(F.col("seats") <= 10, "1-10")
         .when(F.col("seats") <= 25, "11-25")
         .when(F.col("seats") <= 50, "26-50")
         .otherwise("51+")
    )
)

# Churn by customer size

churn_by_seats = (
    customer_seat_segments
    .groupBy("seat_range")
    .agg(
        F.countDistinct("account_id").alias("customers"),
        F.sum("churned").alias("churned_customers"),
        F.round(F.avg("churned") * 100, 2).alias("churn_rate")
    )
    .orderBy("seat_range")
)

display(churn_by_seats)

StatementMeta(, 7daffc0c-6948-41c1-bf29-dcb14590e809, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4e862f34-9555-4d1d-b46d-8cdce46b27b1)

### Result

- Customers with **1–10 seats** have the highest churn rate at **23.74%**, with 47 out of 198 customers churned.
- Customers with **11–25 seats** have a churn rate of **20.13%**, with 32 out of 159 customers churned.
- Customers with **26–50 seats** have a churn rate of **21.70%**, with 23 out of 106 customers churned.
- Customers with **51+ seats** have a churn rate of **21.62%**, with 8 out of 37 customers churned.
- The **1–10 seat segment shows the highest churn**, while the larger segments have relatively similar churn rates.

### Business Insight

Smaller customer accounts appear to have somewhat higher churn in this dataset. However, the difference is not extremely large, so seat count alone should not be treated as a strong churn predictor.

Customer size should be considered together with factors such as **industry, acquisition source, plan tier, product engagement, and support experience**.

## 5. Churned vs Retained Customer Comparison

This analysis compares the overall characteristics of churned and retained customers.

We will examine:

- Customer seats
- Subscription MRR
- Product usage
- Support activity
- Customer satisfaction

The objective is to identify meaningful behavioral and commercial differences between churned and retained customers and determine which characteristics may be useful for identifying potential churn-risk patterns.

In [10]:
# Prepare customer-level subscription metrics

subscription_summary = (
    subscriptions
    .groupBy("account_id")
    .agg(
        F.round(F.avg("mrr_amount"), 2).alias("avg_mrr"),
        F.round(F.avg("seats"), 2).alias("avg_subscription_seats")
    )
)

# Prepare customer-level usage metrics

usage_customer = (
    feature_usage
    .join(
        subscriptions.select("subscription_id", "account_id"),
        on="subscription_id",
        how="inner"
    )
    .groupBy("account_id")
    .agg(
        F.sum("usage_count").alias("total_usage"),
        F.count("*").alias("usage_events"),
        F.countDistinct("usage_date").alias("active_usage_days")
    )
)

# Prepare customer-level support metrics

support_customer = (
    support_tickets
    .groupBy("account_id")
    .agg(
        F.count("*").alias("support_tickets"),
        F.round(F.avg("satisfaction_score"), 2).alias("avg_satisfaction"),
        F.round(F.avg("resolution_time_hours"), 2).alias("avg_resolution_time")
    )
)

# Combine customer-level metrics

customer_comparison = (
    accounts
    .select(
        "account_id",
        "seats",
        "churn_flag"
    )
    .withColumn(
        "churned",
        F.when(F.col("churn_flag") == True, 1).otherwise(0)
    )
    .join(subscription_summary, on="account_id", how="left")
    .join(usage_customer, on="account_id", how="left")
    .join(support_customer, on="account_id", how="left")
)

# Compare churned vs retained customers

churn_retained_comparison = (
    customer_comparison
    .groupBy("churned")
    .agg(
        F.countDistinct("account_id").alias("customers"),
        F.round(F.avg("seats"), 2).alias("avg_seats"),
        F.round(F.avg("avg_mrr"), 2).alias("avg_mrr"),
        F.round(F.avg("total_usage"), 2).alias("avg_total_usage"),
        F.round(F.avg("usage_events"), 2).alias("avg_usage_events"),
        F.round(F.avg("active_usage_days"), 2).alias("avg_active_usage_days"),
        F.round(F.avg("support_tickets"), 2).alias("avg_support_tickets"),
        F.round(F.avg("avg_satisfaction"), 2).alias("avg_satisfaction"),
        F.round(F.avg("avg_resolution_time"), 2).alias("avg_resolution_time")
    )
    .orderBy("churned")
)

display(churn_retained_comparison)

StatementMeta(, 7daffc0c-6948-41c1-bf29-dcb14590e809, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 70224046-c4f8-4ddf-88c4-e44a3b9a6dd1)

### Result

- Retained customers had a higher average seat count of **20.93**, compared with **19.24** for churned customers.
- Retained customers had a higher average MRR of **$2,315.30**, compared with **$2,082.92** for churned customers.
- Interestingly, churned customers showed slightly higher product engagement:
  - Average total usage: **522.04** vs **495.13**
  - Average usage events: **52.05** vs **49.42**
  - Average active usage days: **50.11** vs **47.69**
- Support activity was very similar between the two groups:
  - Average support tickets: **4.00** for churned vs **4.08** for retained customers.
  - Average satisfaction: **4.00** for churned vs **3.95** for retained customers.
  - Average resolution time: **35.49 hours** for churned vs **36.45 hours** for retained customers.

### Business Insight

The comparison shows that **simple usage volume and support metrics do not clearly distinguish churned customers from retained customers** in this dataset.

Churned customers actually show slightly higher product usage, while retained customers have somewhat higher MRR and seat counts. This suggests that churn risk may depend on a **combination of customer characteristics and behavioral patterns**, rather than a single metric.

These findings will be used to identify more specific **potential churn-risk patterns** in the next analysis.


## 6. Potential Churn-Risk Patterns

This analysis combines customer-level behavioral, commercial, and support metrics to identify potential patterns associated with churn.

We will examine the relationship between churn and:

- Customer seats
- Average MRR
- Product usage
- Usage frequency
- Active usage days
- Support ticket volume
- Customer satisfaction
- Support resolution time

The objective is to identify variables that may be useful as potential churn-risk indicators.

These patterns represent associations in the dataset and should not be interpreted as proof of causation.

In [11]:
# Calculate correlations between churn and customer-level numeric metrics

risk_correlation = (
    customer_comparison
    .select(
        "churned",
        "seats",
        "avg_mrr",
        "total_usage",
        "usage_events",
        "active_usage_days",
        "support_tickets",
        "avg_satisfaction",
        "avg_resolution_time"
    )
)

# Calculate correlation of each metric with churn

correlation_results = []

metrics = [
    "seats",
    "avg_mrr",
    "total_usage",
    "usage_events",
    "active_usage_days",
    "support_tickets",
    "avg_satisfaction",
    "avg_resolution_time"
]

for metric in metrics:
    correlation = risk_correlation.stat.corr("churned", metric)
    correlation_results.append((metric, correlation))

correlation_df = spark.createDataFrame(
    correlation_results,
    ["metric", "correlation_with_churn"]
)

correlation_df = correlation_df.withColumn(
    "correlation_with_churn",
    F.round("correlation_with_churn", 4)
).orderBy(
    F.desc(F.abs(F.col("correlation_with_churn")))
)

display(correlation_df)

StatementMeta(, 7daffc0c-6948-41c1-bf29-dcb14590e809, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7b3b8477-47b1-4f9c-8b86-9b70da006d44)

### Result

The correlation analysis shows that all examined variables have relatively weak correlations with churn.

- **Total usage** has the strongest positive correlation with churn at **0.0642**.
- **Usage events** have a correlation of **0.0633** with churn.
- **Active usage days** have a correlation of **0.0619** with churn.
- **Average MRR** has a weak negative correlation of **-0.0602** with churn.
- **Average resolution time** has a correlation of **-0.0346**.
- **Seats** have a correlation of **-0.0334**.
- **Support tickets** have a correlation of **-0.0204**.
- **Average satisfaction** has almost no correlation with churn at **0.0078**.

### Business Insight

None of the analyzed numerical variables shows a strong linear relationship with churn. This suggests that **customer churn is likely influenced by a combination of factors rather than a single measurable variable**.

The analysis highlights the importance of considering multiple dimensions — such as **industry, referral source, plan tier, customer size, product engagement, and support experience** — when investigating churn risk.

These correlations indicate **associations, not causation**, and should be validated with additional analysis before being used for business decisions.


## 7. Product Usage by Plan

This analysis examines how product engagement differs across subscription plan tiers.

We will compare:

- Total usage count
- Usage events
- Active usage days
- Number of active subscriptions

across **Basic, Pro, and Enterprise** plans.

The objective is to understand whether customers on different plans demonstrate different levels of product engagement and identify whether higher-tier plans show stronger or weaker usage patterns.

In [3]:
# Product usage by plan tier

usage_by_plan = (
    feature_usage
    .join(
        subscriptions.select(
            "subscription_id",
            "plan_tier"
        ),
        on="subscription_id",
        how="inner"
    )
    .groupBy("plan_tier")
    .agg(
        F.sum("usage_count").alias("total_usage_count"),
        F.count("*").alias("usage_events"),
        F.countDistinct("usage_date").alias("active_usage_days"),
        F.countDistinct("subscription_id").alias("active_subscriptions")
    )
    .orderBy(F.desc("total_usage_count"))
)

display(usage_by_plan)

StatementMeta(, 76b53e30-46d3-4735-a804-8843b5835bc4, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 16589e63-5be1-497b-a2c9-ca12d7ff2bc9)

### Result

- **Enterprise** subscriptions recorded the highest total usage count at **86,076** and the highest number of usage events at **8,580**.
- **Pro** subscriptions recorded **84,561** total usage count and **8,440** usage events.
- **Basic** subscriptions recorded **79,888** total usage count and **7,980** usage events.
- Enterprise also has the highest number of active subscriptions in the usage data (**1,714**), followed by Pro (**1,665**) and Basic (**1,588**).
- Overall usage volume is higher for Enterprise customers, followed by Pro and Basic customers.

### Business Insight

Enterprise customers generate the highest overall product usage, followed by Pro and Basic customers. However, these totals are influenced by the number of subscriptions in each plan, so total usage alone should not be interpreted as higher engagement per customer.

The `active_usage_days` metric is not used for plan-level comparison because it represents distinct usage dates across the entire dataset rather than usage days per plan.


## 8. Feature Error Rates

This analysis examines product reliability by measuring the error rate associated with each feature.

We will compare features based on:

- Total usage count
- Total errors
- Error rate

The error rate will be calculated as:

**Error Rate = Total Errors / Total Usage Count × 100**

The objective is to identify features with relatively higher error rates and determine which areas of the product may require further investigation.

In [4]:
# Calculate error rates by feature

feature_error_analysis = (
    feature_usage
    .groupBy("feature_name")
    .agg(
        F.sum("usage_count").alias("total_usage_count"),
        F.sum("error_count").alias("total_errors")
    )
    .withColumn(
        "error_rate",
        F.round(
            F.col("total_errors") / F.col("total_usage_count") * 100,
            2
        )
    )
    .orderBy(F.desc("error_rate"))
)

display(feature_error_analysis)

StatementMeta(, 76b53e30-46d3-4735-a804-8843b5835bc4, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5f7001be-4494-49c5-82eb-68301ff81eb6)

### Result

- **feature_4** has the highest error rate at **6.56%**, with 418 errors from 6,374 usage counts.
- **feature_9** has the second-highest error rate at **6.51%**, with 404 errors from 6,207 usage counts.
- **feature_26** follows with an error rate of **6.45%**, with 417 errors from 6,470 usage counts.
- Other relatively high-error features include **feature_16 (6.31%)**, **feature_18 (6.31%)**, and **feature_2 (6.15%)**.
- **feature_8** and **feature_20** have the lowest observed error rate at **4.71%**.
- Overall, error rates vary across features, indicating differences in product reliability or error frequency between features.

### Business Insight

Features with higher error rates may require additional investigation by the product or engineering teams. In particular, **feature_4, feature_9, and feature_26** show comparatively higher error rates and could be prioritized for further diagnostic analysis.

A high error rate does not by itself establish that a feature causes customer churn; it should be analyzed alongside usage, customer impact, and churn behavior.


## 9. Beta Feature Adoption

This analysis examines the adoption and usage of beta features within the product.

We will compare beta and non-beta features based on:

- Number of features
- Usage events
- Total usage count
- Number of subscriptions using the features
- Average usage per event

The objective is to understand how widely beta features are being adopted and whether customers are actively engaging with newly introduced product capabilities.

In [5]:
# Analyze beta feature adoption

beta_feature_adoption = (
    feature_usage
    .groupBy("is_beta_feature")
    .agg(
        F.countDistinct("feature_name").alias("features"),
        F.count("*").alias("usage_events"),
        F.sum("usage_count").alias("total_usage_count"),
        F.countDistinct("subscription_id").alias("active_subscriptions"),
        F.round(F.avg("usage_count"), 2).alias("avg_usage_per_event")
    )
    .orderBy("is_beta_feature")
)

display(beta_feature_adoption)

StatementMeta(, 76b53e30-46d3-4735-a804-8843b5835bc4, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 74655020-ea7b-42fa-b922-2dcbf735fd81)

### Result

- The dataset contains **40 features marked as beta** and **40 non-beta features**.
- Non-beta features generated **22,456 usage events** and a total usage count of **224,939**.
- Beta features generated **2,544 usage events** and a total usage count of **25,586**.
- Beta features were used by **2,023 subscriptions**, compared with **4,946 subscriptions** for non-beta features.
- Average usage per event was very similar between beta and non-beta features:
  - Beta features: **10.06**
  - Non-beta features: **10.02**
- Beta features therefore represent a smaller share of overall product activity, but they still show meaningful adoption across the subscription base.

### Business Insight

Beta features have substantially lower overall usage volume than non-beta features, but adoption across **2,023 subscriptions** indicates that they are being actively explored by a significant portion of the customer base.

The similar average usage per event suggests that when beta features are used, their usage intensity is comparable to non-beta features. This can help product teams identify opportunities for further beta testing, improvement, and eventual promotion to standard features.


## 10. Support Workload & Resolution by Plan

This analysis examines how customer support workload and service performance differ across subscription plan tiers.

We will compare:

- Total support tickets
- Average tickets per customer
- Average resolution time
- Average first-response time
- Average satisfaction score
- Escalated tickets

across **Basic, Pro, and Enterprise** plans.

The objective is to understand which subscription plans generate greater support workload and whether support performance differs across customer segments.

In [6]:
# Support workload and resolution by plan tier

support_plan_analysis = (
    support_tickets
    .join(
        subscriptions.select(
            "account_id",
            "plan_tier"
        ).dropDuplicates(["account_id"]),
        on="account_id",
        how="inner"
    )
    .groupBy("plan_tier")
    .agg(
        F.count("*").alias("total_tickets"),
        F.countDistinct("account_id").alias("customers"),
        F.round(
            F.count("*") / F.countDistinct("account_id"),
            2
        ).alias("avg_tickets_per_customer"),
        F.round(
            F.avg("resolution_time_hours"),
            2
        ).alias("avg_resolution_time"),
        F.round(
            F.avg("first_response_time_minutes"),
            2
        ).alias("avg_first_response_time"),
        F.round(
            F.avg("satisfaction_score"),
            2
        ).alias("avg_satisfaction"),
        F.sum(
            F.when(F.col("escalation_flag") == True, 1).otherwise(0)
        ).alias("escalated_tickets")
    )
    .orderBy(F.desc("total_tickets"))
)

display(support_plan_analysis)

StatementMeta(, 76b53e30-46d3-4735-a804-8843b5835bc4, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 12dd010d-7f11-48c0-9219-a0465c79b1d8)

### Result

- **Enterprise** generated the highest support workload with **740 tickets** across 183 customers.
- **Pro** generated **689 tickets** across 172 customers.
- **Basic** generated **571 tickets** across 137 customers.
- Average tickets per customer were relatively similar:
  - Basic: **4.17**
  - Enterprise: **4.04**
  - Pro: **4.01**
- **Basic** customers had the longest average resolution time at **36.76 hours**, followed by Enterprise at **35.64 hours** and Pro at **35.35 hours**.
- Average first-response times were also relatively similar, ranging from **87.57 to 89.72 minutes**.
- Average satisfaction scores were nearly identical across plans, ranging from **3.98 to 3.99**.
- Escalated tickets were **34 for Enterprise**, **31 for Basic**, and **30 for Pro**.

### Business Insight

Enterprise customers generate the highest overall support workload, largely reflecting their larger customer base in the support data. However, **support workload per customer is relatively similar across plans**.

Basic customers have the longest average resolution time and the highest tickets-per-customer rate, while satisfaction remains broadly consistent across all plans. This suggests that support experience should be evaluated using multiple measures rather than ticket volume alone.


## 11. Churn Reasons

This analysis examines the reasons recorded for customer churn events.

We will analyze:

- Churn reason
- Number of churn events
- Share of total churn events
- Refund amount associated with each reason

The objective is to identify the major recorded reasons for churn and understand which churn reasons are associated with higher financial impact.

In [7]:
# Analyze churn reasons

churn_reason_analysis = (
    churn_events
    .groupBy("reason_code")
    .agg(
        F.count("*").alias("churn_events"),
        F.round(F.sum("refund_amount_usd"), 2).alias("total_refund_amount"),
        F.round(F.avg("refund_amount_usd"), 2).alias("avg_refund_amount")
    )
)

# Calculate percentage of total churn events

total_churn_events = churn_events.count()

churn_reason_analysis = (
    churn_reason_analysis
    .withColumn(
        "event_share_pct",
        F.round(
            F.col("churn_events") / total_churn_events * 100,
            2
        )
    )
    .orderBy(F.desc("churn_events"))
)

display(churn_reason_analysis)

StatementMeta(, 76b53e30-46d3-4735-a804-8843b5835bc4, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 841e0e3f-a2df-4f00-840a-985e80c89a3a)

### Result

- **Features** is the most frequently recorded churn reason, accounting for **114 churn events (19.00%)**.
- **Support** and **Budget** each account for **104 churn events (17.33%)**.
- **Unknown** accounts for **95 churn events (15.83%)**, indicating that some churn events lack a clearly identified reason.
- **Competitor** accounts for **92 events (15.33%)**, while **Pricing** accounts for **91 events (15.17%)**.
- The highest total refund amount is associated with **Features ($1,905.90)**, followed by **Unknown ($1,742.39)**.

### Business Insight

Product-related issues are the most frequently recorded churn reason, followed by support and budget constraints. The relatively large **Unknown** category also suggests that improving churn-reason tracking could provide better insight into customer attrition.

## 12. Churn Events and Customer Lifecycle Changes

This analysis examines whether churn events are associated with recent subscription changes.

We will compare churn events involving:

- Preceding upgrades
- Preceding downgrades
- Reactivation events

The objective is to understand whether subscription changes and customer lifecycle events are commonly observed around churn.

In [8]:
# Analyze churn events associated with upgrades, downgrades, and reactivation

total_churn_events = churn_events.count()

lifecycle_analysis = (
    churn_events
    .select(
        F.sum(
            F.when(F.col("preceding_upgrade_flag") == True, 1).otherwise(0)
        ).alias("upgrade_related_events"),

        F.sum(
            F.when(F.col("preceding_downgrade_flag") == True, 1).otherwise(0)
        ).alias("downgrade_related_events"),

        F.sum(
            F.when(F.col("is_reactivation") == True, 1).otherwise(0)
        ).alias("reactivation_events")
    )
    .withColumn("total_churn_events", F.lit(total_churn_events))
    .withColumn(
        "upgrade_event_pct",
        F.round(
            F.col("upgrade_related_events") / F.col("total_churn_events") * 100,
            2
        )
    )
    .withColumn(
        "downgrade_event_pct",
        F.round(
            F.col("downgrade_related_events") / F.col("total_churn_events") * 100,
            2
        )
    )
    .withColumn(
        "reactivation_event_pct",
        F.round(
            F.col("reactivation_events") / F.col("total_churn_events") * 100,
            2
        )
    )
)

display(lifecycle_analysis)

StatementMeta(, 76b53e30-46d3-4735-a804-8843b5835bc4, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7cc28f96-39f0-4c3f-a2a9-cb8897f50a73)

### Result

- The dataset contains **600 total churn events**.
- **123 churn events (20.5%)** were preceded by an upgrade.
- **53 churn events (8.83%)** were preceded by a downgrade.
- **61 churn events (10.17%)** were associated with reactivation events.
- Upgrade-related events were more common than downgrade-related events among the recorded churn events.

### Business Insight

A noticeable share of churn events occurred after customer plan changes, particularly upgrades. This indicates that customer lifecycle changes are worth considering when investigating churn patterns. However, these relationships should be treated as associations rather than causal effects.

## 13. Potential Churn-Risk Patterns

This final analysis combines key customer characteristics and behavioral indicators to identify patterns associated with churn.

We will examine:

- Product usage
- Support activity
- Customer tenure
- MRR
- Seats
- Plan tier

The objective is to identify customer characteristics that are associated with churn and summarize potential churn-risk patterns for further investigation.

In [9]:
# Create a customer-level feature set

usage_customer = (
    feature_usage
    .join(
        subscriptions.select("subscription_id", "account_id"),
        on="subscription_id",
        how="inner"
    )
    .groupBy("account_id")
    .agg(
        F.sum("usage_count").alias("total_usage"),
        F.count("*").alias("usage_events"),
        F.countDistinct("usage_date").alias("active_usage_days")
    )
)

support_customer = (
    support_tickets
    .groupBy("account_id")
    .agg(
        F.count("*").alias("support_tickets"),
        F.round(F.avg("satisfaction_score"), 2).alias("avg_satisfaction"),
        F.round(F.avg("resolution_time_hours"), 2).alias("avg_resolution_time"),
        F.sum(
            F.when(F.col("escalation_flag") == True, 1).otherwise(0)
        ).alias("escalated_tickets")
    )
)

customer_features = (
    accounts
    .join(usage_customer, on="account_id", how="left")
    .join(support_customer, on="account_id", how="left")
    .join(
        subscriptions.groupBy("account_id").agg(
            F.round(F.avg("mrr_amount"), 2).alias("avg_mrr"),
            F.round(F.avg("seats"), 2).alias("avg_seats")
        ),
        on="account_id",
        how="left"
    )
    .withColumn(
        "tenure_days",
        F.datediff(
            F.current_date(),
            F.to_date("signup_date")
        )
    )
    .withColumn(
        "churned",
        F.when(F.col("churn_flag") == True, 1).otherwise(0)
    )
    .fillna(0)
)

# Compare key characteristics between churned and retained customers

risk_pattern_analysis = (
    customer_features
    .groupBy("churned")
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("avg_mrr"), 2).alias("avg_mrr"),
        F.round(F.avg("avg_seats"), 2).alias("avg_seats"),
        F.round(F.avg("total_usage"), 2).alias("avg_total_usage"),
        F.round(F.avg("usage_events"), 2).alias("avg_usage_events"),
        F.round(F.avg("active_usage_days"), 2).alias("avg_active_usage_days"),
        F.round(F.avg("support_tickets"), 2).alias("avg_support_tickets"),
        F.round(F.avg("avg_satisfaction"), 2).alias("avg_satisfaction"),
        F.round(F.avg("avg_resolution_time"), 2).alias("avg_resolution_time"),
        F.round(F.avg("tenure_days"), 2).alias("avg_tenure_days")
    )
    .orderBy("churned")
)

display(risk_pattern_analysis)

StatementMeta(, 76b53e30-46d3-4735-a804-8843b5835bc4, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 223a2386-048d-41d4-8bef-04691d88c0fa)

### Result

- **390 customers were retained**, while **110 customers were churned**.
- Churned customers had a lower average MRR (**2,082.92**) compared with retained customers (**2,315.30**).
- Churned customers had slightly fewer average seats (**28.49**) than retained customers (**30.41**).
- Churned customers showed slightly higher product engagement:
  - Average total usage: **522.04** vs **495.13**
  - Average usage events: **52.05** vs **49.42**
  - Average active usage days: **50.11** vs **47.69**
- Average support activity was broadly similar:
  - Support tickets: **3.93** for churned vs **4.02** for retained customers.
  - Satisfaction: **3.71** for churned vs **3.69** for retained customers.
  - Resolution time: **34.84 hours** for churned vs **35.89 hours** for retained customers.
- Average customer tenure was also relatively similar, with churned customers at approximately **953 days** and retained customers at approximately **996 days**.

### Business Insight

The analysis does **not** show that lower product usage or poorer support satisfaction is clearly associated with churn. In fact, churned customers show slightly higher usage activity than retained customers.

This suggests that **churn at RavenStack may involve multiple factors rather than simple low engagement or poor support**. Revenue level, plan characteristics, customer segment, lifecycle events, and specific churn reasons should therefore be considered together when investigating retention.

These findings represent **associations in the dataset, not causal relationships**.
